# Importing

In [ ]:
! source /home/ivm/envs/valid_env/bin/activate
import polars as pl
import numpy as np
import sys
sys.path.append(("/home/ivm/valid/scripts/utils/"))
sys.path.append(("/home/ivm/valid/scripts/run_files/checks/"))

from general_utils import *
from selection_checks_utils import *

%load_ext autoreload
%autoreload 2

# File paths

In [ ]:
past_preds_paths = {
        "eGFR": "/home/ivm/valid/data/processed_data/kanta_R14/step5_predict/v17_2023val2024_w3/y_MEAN_ABNORM/xgb_logloss_5_all_pgs/models/egfr/2026-04-27/preds_2026-04-27.parquet",
        "HbA1c": "/home/ivm/valid/data/processed_data/kanta_R14/step5_predict/v10_2023val2024_w3/y_MEAN_ABNORM/xgb_logloss_5_all_pgs/models/hba1c/2026-04-24/preds_2026-04-24.parquet",
        "TSH": "/home/ivm/valid/data/processed_data/kanta_R14/step5_predict/v7_2023val2024_w3/y_MEAN_ABNORM/xgb_mlogloss_5_all_pgs/models/tsh/2026-04-27/preds_2026-04-27.parquet",
        "LDL": "/home/ivm/valid/data/processed_data/kanta_R14/step5_predict/v7_2023val2024_w3/y_MEAN_ABNORM/xgb_logloss_5_all_pgs/models/ldl/2026-04-24/preds_2026-04-24.parquet"
    }

In [ ]:
future_preds_paths = {
    "eGFR": "/home/ivm/valid/data/processed_data/kanta_R14/step5_predict/v17_2023val2024_w3/y_MEAN_ABNORM/xgb_logloss_final_5_all_pgs/models/egfr/2026-04-27/final_preds_2026-04-27.parquet",
    "HbA1c":"/home/ivm/valid/data/processed_data/kanta_R14/step5_predict/v10_2023val2024_w3/y_MEAN_ABNORM/xgb_logloss_final_5_all_pgs/models/hba1c/2026-04-24/final_preds_2026-04-24.parquet",
    "LDL": "/home/ivm/valid/data/processed_data/kanta_R14/step5_predict/v7_2023val2024_w3/y_MEAN_ABNORM/xgb_logloss_final_5_all_pgs/models/ldl/2026-04-27/final_preds_2026-04-27.parquet",
    "TSH":"/home/ivm/valid/data/processed_data/kanta_R14/step5_predict/v7_2023val2024_w3/y_MEAN_ABNORM/xgb_mlogloss_final_5_all_pgs/models/tsh/2026-04-27/final_preds_2026-04-27.parquet"
}

# Cuts and sample sizes

In [ ]:
min_age = 30

In [ ]:
from collections import defaultdict

min_probs = {"eGFR": 0.0025, "HbA1c": 0.0025, "LDL": 0.015, "TSH": 0.005}
all_quart_cuts = defaultdict(list)

for lab_name in future_preds_paths:
    new_quart_cuts = np.arange(0.005, 1.0, 0.01).round(3).tolist()

    if min_probs[lab_name]<min(new_quart_cuts):
        new_quart_cuts = [min_probs[lab_name]]+new_quart_cuts
    elif min_probs[lab_name]>min(new_quart_cuts):
        new_quart_cuts = new_quart_cuts[1:]
        
    crnt_quart_cuts = []
    count = 0
    last_possible_sample_size = 0
    crnt_max_prob = 1.0
    
    while(crnt_quart_cuts != new_quart_cuts):
        crnt_quart_cuts = new_quart_cuts
        max_n = get_max_n(future_preds_paths[lab_name], 
                          crnt_quart_cuts, 
                          min_probs[lab_name],
                          1,
                          bb_specific=True,
                          min_age=min_age)
        new_quart_cuts, last_possible_sample_size, crnt_max_prob = check_sample_sizes(crnt_quart_cuts,
                                                                                      max_n,
                                                                                      last_possible_sample_size, 
                                                                                      crnt_max_prob,
                                                                                      vocal=False)
        count += 1
    
    all_quart_cuts[lab_name] = new_quart_cuts[:-1]+[crnt_max_prob]
    print("# # # # # # # # "+lab_name+" # # # # # # # # # # # #")
    print(all_quart_cuts[lab_name])
    print()

###  Sample sizes

In [ ]:
all_sample_sizes = defaultdict(dict)
total_n = 500
for lab_name in future_preds_paths:
    max_n = get_max_n(future_preds_paths[lab_name], 
                      all_quart_cuts[lab_name], 
                      min_probs[lab_name],
                      all_quart_cuts[lab_name][-1],
                      bb_specific=True,
                     min_age=min_age)
    #### Round 1
    crnt_sample_sizes={idx+1: (total_n/len(all_quart_cuts[lab_name])/2)*2 for idx in range(len(all_quart_cuts[lab_name]))}
    crnt_sample_sizes, reduce_idxs, rest_idxs = reduce_sample_sizes(crnt_sample_sizes, max_n)
    sample_size_reduce_idxs = np.asarray([crnt_sample_size for idx, crnt_sample_size in crnt_sample_sizes.items() if idx in reduce_idxs]).sum()
    
    distribute_left_size = round((total_n-sample_size_reduce_idxs)/2/len(rest_idxs))*2
    
    #### Round 2
    for idx, crnt_sample_size  in crnt_sample_sizes.items():
        if idx in rest_idxs:
            crnt_sample_sizes[idx] = distribute_left_size
    
    crnt_sample_sizes, reduce_idxs_2, rest_idxs_2 = reduce_sample_sizes(crnt_sample_sizes, max_n)
    
    reduce_idxs = list(set(reduce_idxs + reduce_idxs_2))
    rest_idxs = [crnt_idx for crnt_idx in range(1,len(all_quart_cuts[lab_name])+1) if crnt_idx not in reduce_idxs]
    sample_size_reduce_idxs = np.asarray([crnt_sample_size for idx, crnt_sample_size in crnt_sample_sizes.items() if idx in reduce_idxs]).sum()    

    distribute_left_size = round((total_n-sample_size_reduce_idxs)/2/len(rest_idxs))*2
    
    #### Final
    for idx, crnt_sample_size  in crnt_sample_sizes.items():
        if idx in rest_idxs:
            crnt_sample_sizes[idx] = distribute_left_size
    
    crnt_sample_sizes, reduce_idxs, rest_idxs = reduce_sample_sizes(crnt_sample_sizes, max_n)

    all_sample_sizes[lab_name] = crnt_sample_sizes
    
    print("# # # # # # # # "+lab_name+" # # # # # # # # # # # #")
    print(np.asarray([value for key, value in crnt_sample_sizes.items()]).sum())
    print(crnt_sample_sizes)
    print()
    
print("# # # # # # # Total # # # # # # # # # ")
print(np.asarray([value for key, value in crnt_sample_sizes.items() for lab_name, crnt_sample_sizes in all_sample_sizes.items()]).sum())

# Drawing

## eGFR

In [ ]:
draw_now(past_preds_paths, 
         future_preds_paths,
         all_quart_cuts, 
         all_sample_sizes,
         min_probs=min_probs, 
         part_pct=0.25, 
         lab_name="eGFR", 
         n_boots=100,
         min_age=min_age,
         save_files=True)

## HbA1c

In [ ]:
draw_now(past_preds_paths, 
         future_preds_paths,
         all_quart_cuts, 
         all_sample_sizes,
         min_probs=min_probs, 
         part_pct=0.25, 
         lab_name="HbA1c", 
         n_boots=100,
         min_age=min_age,
        save_files=True)

## TSH

In [ ]:
draw_now(past_preds_paths, 
         future_preds_paths,
         all_quart_cuts, 
         all_sample_sizes,
         min_probs=min_probs, 
         part_pct=0.25, 
         lab_name="TSH", 
         n_boots=100,
        min_age=min_age,
        save_files=True)

## LDL

In [ ]:
draw_now(past_preds_paths, 
         future_preds_paths,
         all_quart_cuts, 
         all_sample_sizes,
         min_probs=min_probs, 
         part_pct=0.25, 
         lab_name="LDL", 
         n_boots=100,
         min_age=min_age,
        save_files=True)

# Collapsed bins

###  Settings

In [ ]:
all_collapsed_quart_cuts = defaultdict(list)
all_collapsed_sample_sizes = defaultdict(list)
part_pct = 0.25
for lab_name in all_quart_cuts:
    all_collapsed_quart_cuts[lab_name] = [0.015, 0.025, 0.075, 0.125, 0.175, 0.225,  all_quart_cuts[lab_name][-1]]
    
    all_collapsed_sample_sizes[lab_name] = collapse_sample_sizes(all_collapsed_quart_cuts[lab_name],
                                                               all_quart_cuts[lab_name],
                                                               all_sample_sizes[lab_name],
                                                               min_probs[lab_name])
    print(np.asarray([value for key, value in all_collapsed_sample_sizes[lab_name].items()]).sum())
    print(all_collapsed_sample_sizes[lab_name])
    print({key: value*all_collapsed_quart_cuts[lab_name][int(key-1)]*part_pct for key, value in all_collapsed_sample_sizes[lab_name].items()})
    print(all_collapsed_quart_cuts[lab_name])

## eGFR

In [ ]:
draw_now(past_preds_paths, 
             future_preds_paths,
             all_collapsed_quart_cuts, 
             all_collapsed_sample_sizes,
             min_probs=min_probs, 
             part_pct=0.25, 
             lab_name="eGFR", 
             n_boots=100,
             base_path="/home/ivm/valid/data/processed_data/kanta_R14/model_evals/selections/2026-04-28/smaller_sections/",
             save_files=True)

## HbA1c

In [ ]:
draw_now(past_preds_paths, 
             future_preds_paths,
             all_collapsed_quart_cuts, 
             all_collapsed_sample_sizes,
             min_probs=min_probs, 
             part_pct=0.25, 
             lab_name="HbA1c", 
             n_boots=100,
             base_path="/home/ivm/valid/data/processed_data/kanta_R14/model_evals/selections/2026-04-28/bigger_sections/",
            save_files=True)

## TSH

In [ ]:
draw_now(past_preds_paths, 
             future_preds_paths,
             all_collapsed_quart_cuts, 
             all_collapsed_sample_sizes,
             min_probs=min_probs, 
             part_pct=0.25, 
             lab_name="TSH", 
             n_boots=100,
             base_path="/home/ivm/valid/data/processed_data/kanta_R14/model_evals/selections/2026-04-28/bigger_sections/",
            save_files=True)

## LDL

In [ ]:
draw_now(past_preds_paths, 
             future_preds_paths,
             all_collapsed_quart_cuts, 
             all_collapsed_sample_sizes,
             min_probs=min_probs, 
             part_pct=0.25, 
             lab_name="LDL", 
             n_boots=100,
             base_path="/home/ivm/valid/data/processed_data/kanta_R14/model_evals/selections/2026-04-28/bigger_sections/",
            save_files=True)